# MOMO Colab Quick Trial

This notebook is the lightweight path for trying MOMO without installing anything locally. It runs Whisper, Ollama, and the MOMO CLI inside a temporary Google Colab GPU runtime.

For private recordings, long meetings, or repeated use, use the local/server GUI instead.

## Before running

1. Choose **Runtime → Change runtime type → GPU**.
2. Run every cell from top to bottom.
3. Upload one meeting recording when prompted.
4. Edit the topic-details cell if you want meeting-specific focus terms.

The notebook intentionally fails if CUDA is unavailable. MOMO is designed to use GPU for this trial path.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile

REPO_URL = "https://github.com/jjunsss/MOMO.git"
BRANCH = "main"

# Trial-stable defaults for Colab. For a slower quality pass, use:
# ASR_MODEL = "large-v3", SUMMARY_MODE = "thorough", ENABLE_CRITIQUE = "true".
ASR_MODEL = "medium"
LLM_MODEL = "qwen3.5:9b"
LLM_NUM_CTX = "8192"
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu124"
SUMMARY_MODE = "fast"
ENABLE_CRITIQUE = "false"
OUTPUT_LANGUAGE = "ko"  # "ko" or "en"

PROJECT_DIR = Path("/content/MOMO")
WORKSPACE_DIR = Path("/content/momo_workspace")
VIDEOS_DIR = WORKSPACE_DIR / "videos"
RUNS_DIR = WORKSPACE_DIR / "runs"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"


def run(command, *, cwd=None, env=None):
    if isinstance(command, (list, tuple)):
        printable = " ".join(str(part) for part in command)
    else:
        printable = str(command)
    print(f"$ {printable}", flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True, shell=isinstance(command, str))


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Colab GPU is OFF or unavailable. Use Runtime > Change runtime type > GPU, "
        "then restart the runtime and rerun this notebook."
    )

print("CUDA GPU:", torch.cuda.get_device_name(0))
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi is not available, but PyTorch can see CUDA.")


In [ ]:
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg curl git")

if PROJECT_DIR.exists():
    run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
    run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)])

run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchaudio", "--index-url", TORCH_INDEX_URL])
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[asr]"], cwd=PROJECT_DIR)


In [ ]:
def ollama_ready():
    try:
        with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=2):
            return True
    except (OSError, TimeoutError, urllib.error.URLError):
        return False


if shutil.which("ollama") is None:
    run("curl -fsSL https://ollama.com/install.sh | sh")

if not ollama_ready():
    ollama_env = os.environ.copy()
    ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
    log_file = open("/tmp/momo_ollama.log", "ab")
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT, env=ollama_env)
    for _ in range(90):
        if ollama_ready():
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama did not start. Check /tmp/momo_ollama.log, restart the runtime, and rerun.")

run(["ollama", "pull", LLM_MODEL])


In [ ]:
from google.colab import files

VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No recording was uploaded.")

saved_files = []
for name, data in uploaded.items():
    destination = VIDEOS_DIR / Path(name).name
    destination.write_bytes(data)
    saved_files.append(destination)

print("Uploaded:")
for path in saved_files:
    print("-", path)


In [ ]:
TOPIC_DETAILS = {
    "title": "Colab MOMO Trial",
    "custom_instruction": "결정사항, 액션 아이템, 리스크, 다음 미팅을 우선 정리해 주세요.",
    "topics": [
        "핵심 논의",
        "결정사항",
        "해야 할 일",
        "다음 미팅",
    ],
    "must_check": [
        "잠정 결정과 확정 결정을 구분한다",
        "기술명, 논문명, 모델명처럼 영어로 말한 고유명사는 영어 원문 표기를 우선한다",
        "언급되지 않은 담당자, 날짜, 다음 회의는 추정하지 않는다",
    ],
    "output_language": OUTPUT_LANGUAGE,
}

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
topic_path = WORKSPACE_DIR / "topic_details.json"
topic_path.write_text(json.dumps(TOPIC_DETAILS, ensure_ascii=False, indent=2), encoding="utf-8")
print(topic_path.read_text(encoding="utf-8"))


In [ ]:
env = os.environ.copy()
env.update(
    {
        "MOMO_ASR_DEVICE": "cuda",
        "MOMO_ASR_MODEL": ASR_MODEL,
        "MOMO_LLM_PROVIDER": "ollama",
        "MOMO_LLM_MODEL": LLM_MODEL,
        "MOMO_LLM_BASE_URL": OLLAMA_BASE_URL,
        "MOMO_LLM_NUM_CTX": LLM_NUM_CTX,
        "MOMO_LLM_REQUEST_TIMEOUT_SECONDS": "1800",
        "MOMO_LLM_SUMMARY_MODE": SUMMARY_MODE,
        "MOMO_ENABLE_CRITIQUE": ENABLE_CRITIQUE,
        "MOMO_OUTPUT_LANGUAGE": OUTPUT_LANGUAGE,
        "OLLAMA_HOST": "127.0.0.1:11434",
    }
)

run(
    [
        sys.executable,
        "-m",
        "meeting_ai.cli",
        "auto",
        "--videos-dir",
        str(VIDEOS_DIR),
        "--runs-dir",
        str(RUNS_DIR),
        "--topic-details",
        str(WORKSPACE_DIR / "topic_details.json"),
        "--asr-model",
        ASR_MODEL,
    ],
    cwd=PROJECT_DIR,
    env=env,
)


In [ ]:
from IPython.display import Markdown, display
from google.colab import files

run_dirs = sorted(
    [path for path in RUNS_DIR.iterdir() if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not run_dirs:
    raise RuntimeError("No run output was created.")

latest_run = run_dirs[0]
summary_md = latest_run / "summaries" / "final_summary.md"
if not summary_md.exists():
    raise RuntimeError(f"Final summary was not found: {summary_md}")

print("Latest run:", latest_run)
display(Markdown(summary_md.read_text(encoding="utf-8")))

zip_path = WORKSPACE_DIR / f"{latest_run.name}_momo_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for artifact in [
        summary_md,
        latest_run / "summaries" / "final_summary.json",
        latest_run / "evidence" / "summary_evidence.md",
        latest_run / "transcript" / "normalized_transcript.md",
    ]:
        if artifact.exists():
            archive.write(artifact, artifact.relative_to(latest_run.parent))

files.download(str(zip_path))
